# CS 691 · Module 09 · Finding evidence in text and graphs

Find the records needed to answer where Ava should pick up camera C1, and explain why each record belongs in the answer.

**Before you begin.** You can read a graph edge as a statement about two things. You have used a score to rank choices and a rule to check whether a choice is allowed.

Ava’s reservation names camera C4. One record gives C4’s shelf and another gives its public pickup desk. If Ava is on her way to collect it, which record belongs with the reservation?

<details><summary>Reveal the answer</summary>

The pickup record. It matches both the item, C4, and the relationship the question asks about.

</details>

## What would count as evidence?

Ava asks, “Where do I pick up the camera I reserved?” The desk has hundreds of records. Retrieval means selecting some of those records for the question at hand. First work out what you need to know. Here you need the camera Ava reserved and its pickup location.

A reservation record names Ava and C1. A pickup record names C1 and a desk. The shared camera ID lets you connect the two records even when the second one never mentions Ava.

**Predict before running the cell.** If you kept only the reservation record, which part of Ava’s question could you answer?

In [ ]:
reservations = {"Ava": "C1", "Ben": "C2"}
pickup_desks = {"C1": "West Desk", "C2": "East Desk"}
camera = reservations["Ava"]
print(camera, pickup_desks[camera])

<details><summary>Reveal output and explanation</summary>

```text
C1 West Desk
```

You could identify C1 as her camera. Finding the desk still requires a record about C1’s pickup location.

</details>

## Rank passages by words they share

A passage is a short piece of text. One simple retrieval score counts distinct words that occur in both a query and a passage. In Python, split() separates words at spaces and a set keeps one copy of each word. The & operator keeps the words shared by two sets.

For the query “camera C1 pickup”, P1 names C1 and pickup. P2 names a different camera and mentions pickup. Both passages earn two points. Counting words alone cannot settle which camera Ava requested.

**Predict before running the cell.** If you added “camera” to P1, would a score of three prove that Ava reserved C1?

In [ ]:
query = set("camera c1 pickup".split())
passages = {"P1": "c1 pickup west", "P2": "camera c2 pickup east"}
for record, passage in passages.items():
    words = set(passage.split())
    print(record, len(query & words))

<details><summary>Reveal output and explanation</summary>

```text
P1 2
P2 2
```

No. It would give P1 three matching words. Evidence connecting Ava to C1 still has to come from a reservation record.

</details>

## Follow a relation in the right direction

A graph lookup asks for a particular kind of link. Write a link as an ordered triple containing its starting point, the relation name and its endpoint. To find a pickup desk, first follow Ava’s reserves link to a camera. Start the next lookup at that camera and follow pickup_at.

C1 has separate links for storage and pickup. Reading the relation name matters because a camera can be stored in a vault and collected at a public desk.

**Predict before running the cell.** Suppose Ava reserves C2 and the only pickup link is still for C1. What does this lookup return?

In [ ]:
edges = [("Ava", "reserves", "C1"), ("C1", "stored_at", "Vault"),
         ("C1", "pickup_at", "West Desk")]
for person, relation, camera in edges:
    if (person, relation) == ("Ava", "reserves"):
        for item, link, place in edges:
            if (item, link) == (camera, "pickup_at"):
                print(camera, place)

<details><summary>Reveal output and explanation</summary>

```text
C1 West Desk
```

It prints no desk. The second link must begin at C2 to complete a path from Ava’s reservation.

</details>

## Carry record IDs into the answer

A record ID is a name you can use to find the same record again. Keep each ID with the value it supplied. Another person can then inspect every link in your answer. If one link is missing, say which question remains open.

R4 establishes Ava’s reservation and D8 establishes the desk for C1. Either record alone leaves a piece missing.

**Predict before running the cell.** Another student says the pickup location is East Desk and cites only R4. Which part can you check with R4?

In [ ]:
reservations = {"Ava": ("C1", "R4")}
desks = {"C1": ("West Desk", "D8")}
camera, reservation_id = reservations["Ava"]
if camera in desks:
    desk, desk_id = desks[camera]
    print(f"{camera} at {desk} [{reservation_id}, {desk_id}]")
else:
    print(f"Pickup desk unknown for {camera}")

<details><summary>Reveal output and explanation</summary>

```text
C1 at West Desk [R4, D8]
```

R4 identifies Ava’s reserved item. To settle the disagreement about location, inspect D8, which gives West Desk for that item.

</details>

## Score a passage by its overlapping words

The split call cuts a sentence into words at spaces, and set keeps each different word once. The ampersand finds the words that occur in both the query and the passage. Their count is the passage’s score, so text about the wrong camera can still tie for first.

Replace this line:
query = set("camera c1 pickup".split())
with:
query = set("c1 west".split())
What will print?

In [ ]:
query = set("camera c1 pickup".split())
passages = {"P1": "c1 pickup west", "P2": "camera c2 pickup east"}
for record, passage in passages.items():
    words = set(passage.split())
    print(record, len(query & words))

<details><summary>Worked answer after your change</summary>

P1 2
P2 0

P1 contains both c1 and west. P2 contains neither word.

</details>

## Exercise 1

Mia reserves tripod T2. T2 is stored in Room 1 and collected in Room 4. Write the two graph links needed for “Where do I collect my tripod?”

<details><summary>Hint</summary>

Use the item ID as the endpoint of the first link and the starting point of the second.

</details>

<details><summary>Worked answer</summary>

Mia → reserves → T2 and T2 → pickup_at → Room 4. Following both produces Room 4. The storage link answers a different question.

</details>

## Exercise 2

For the query “pickup lens L2”, P1 says “lens L2 collection” and P2 says “pickup lens L3”. Compute both scores and propose a way to distinguish them.

<details><summary>Hint</summary>

Count shared words ignoring case. Then look for an identifier that names a single item.

</details>

<details><summary>Worked answer</summary>

Each scores two. P1 matches lens and L2. P2 matches pickup and lens. Require the requested item ID before ranking, then inspect whether “collection” describes pickup.

</details>

## Exercise 3

Kai reserves C1 and T2. Both have pickup links. Why could a working graph lookup still give a misleading one-desk answer?

<details><summary>Hint</summary>

Ask how many valid paths begin at Kai and what the question names.

</details>

<details><summary>Worked answer</summary>

Both paths may be valid. If Kai asks specifically for the camera, restrict the item to a camera before following pickup_at. If the item is unspecified, give both item–desk pairs or ask which item Kai means.

</details>

You can now find the records behind an answer. How can you tell whether a sentence a language model writes actually follows from those records?

A draft based on a pickup record says “Collect C4 at South Desk and bring a photo ID.” Which part needs a second kind of evidence?

<details><summary>Reveal the answer</summary>

The ID requirement needs a relevant policy record. The pickup record can support the desk location while leaving that other claim open.

</details>